In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score,
                              precision_score, recall_score)
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load cleaned data
df = pd.read_csv('C:/data/BrandPulse-AI/data/processed/cleaned_tweets.csv')
df = df.dropna(subset=['cleaned_text'])
df = df[df['cleaned_text'].str.strip() != '']

X = df['cleaned_text'].values
y = df['polarity'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Load Classical models
tfidf     = joblib.load('C:/data/BrandPulse-AI/models/tfidf_vectorizer.pkl')
lr_model  = joblib.load('C:/data/BrandPulse-AI/models/logistic_model.pkl')
nb_model  = joblib.load('C:/data/BrandPulse-AI/models/naive_bayes_model.pkl')

# Load LSTM model + tokenizer
lstm_model = load_model('C:/data/BrandPulse-AI/models/lstm_best.h5')
with open('C:/data/BrandPulse-AI/models/tokenizer.json') as f:
    tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(f.read())

print("All models loaded ")
print(f"Test set size: {len(X_test)}")

All models loaded 
Test set size: 318592


In [3]:
MAX_LEN = 100

# Classical predictions
X_test_tfidf = tfidf.transform(X_test)
y_pred_lr = lr_model.predict(X_test_tfidf)
y_pred_nb = nb_model.predict(X_test_tfidf)

# LSTM predictions
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN,
                            padding='post', truncating='post')
y_pred_prob_lstm = lstm_model.predict(X_test_pad, batch_size=512)
y_pred_lstm = (y_pred_prob_lstm > 0.5).astype(int).flatten()

# Calculate metrics for all models
models = {
    'Logistic Regression': y_pred_lr,
    'Naive Bayes':         y_pred_nb,
    'LSTM (BiLSTM)':       y_pred_lstm
}

print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("=" * 60)

results = {}
for name, y_pred in models.items():
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'precision': prec,
                     'recall': rec, 'f1': f1}
    print(f"{name:<25} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

print("=" * 60)

623/623 ━━━━━━━━━━━━━━━━━━━━ 86s 138ms/step
Model                       Accuracy  Precision     Recall         F1
Logistic Regression           0.8058     0.7968     0.8209     0.8087
Naive Bayes                   0.7860     0.7887     0.7812     0.7849
LSTM (BiLSTM)                 0.8104     0.8130     0.8062     0.8096


In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
model_names = list(results.keys())
colors = ['#4ECDC4', '#FF6B6B', '#9B59B6']

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

for i, metric in enumerate(metrics):
    values = [results[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, values, color=colors, width=0.5)
    axes[i].set_title(metric.capitalize(), fontsize=13, fontweight='bold')
    axes[i].set_ylim(0.7, 0.9)
    axes[i].set_ylabel('Score')
    axes[i].tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.002,
                     f'{val:.3f}', ha='center', fontsize=10)

plt.suptitle('Model Comparison — BrandPulse AI',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('C:/data/BrandPulse-AI/reports/figures/model_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved model_comparison.png ")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_preds = [
    ('Logistic Regression', y_pred_lr, 'Blues'),
    ('Naive Bayes', y_pred_nb, 'Oranges'),
    ('LSTM (BiLSTM)', y_pred_lstm, 'Purples')
]

for ax, (name, y_pred, cmap) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Neg', 'Pos'],
                yticklabels=['Neg', 'Pos'], ax=ax)
    ax.set_title(f'{name}\nAccuracy: {acc*100:.2f}%',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('C:/data/BrandPulse-AI/reports/figures/all_confusion_matrices.png',
            dpi=150)
plt.show()
print("Saved all_confusion_matrices.png ")

In [ ]:
import sys
sys.path.append('C:/data/BrandPulse-AI')
from src.preprocessing import clean_tweet

def predict_all_models(tweet):
    cleaned = clean_tweet(tweet)

    # Logistic Regression
    vec = tfidf.transform([cleaned])
    lr_pred = lr_model.predict(vec)[0]
    lr_conf = max(lr_model.predict_proba(vec)[0]) * 100

    # Naive Bayes
    nb_pred = nb_model.predict(vec)[0]
    nb_conf = max(nb_model.predict_proba(vec)[0]) * 100

    # LSTM
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    lstm_prob = lstm_model.predict(pad, verbose=0)[0][0]
    lstm_pred = 1 if lstm_prob > 0.5 else 0
    lstm_conf = max(lstm_prob, 1-lstm_prob) * 100

    label = lambda p: ' POSITIVE' if p == 1 else ' NEGATIVE'

    print(f"Tweet: {tweet}")
    print(f"Cleaned: {cleaned}")
    print(f"{'─'*50}")
    print(f"Logistic Regression : {label(lr_pred)} ({lr_conf:.1f}%)")
    print(f"Naive Bayes         : {label(nb_pred)} ({nb_conf:.1f}%)")
    print(f"LSTM (BiLSTM)       : {label(lstm_pred)} ({lstm_conf:.1f}%)")
    print(f"{'='*50}\n")

# Test tweets
tweets = [
    "The service was absolutely amazing! Best experience ever!",
    "I waited 4 hours just to get a cold burger. Terrible!",
    "Not happy with my purchase. Can't believe this happened.",
    "This product changed my life. Highly recommend to everyone!",
    "Worst customer service I have ever experienced in my life."
]

for tweet in tweets:
    predict_all_models(tweet)

In [ ]:
report = f"""# BrandPulse AI — Model Comparison Report

## Dataset
- Total tweets: 1,592,958
- Train set: 1,274,366
- Test set: 318,592
- Classes: Balanced (50% positive, 50% negative)

## Results

| Model | Accuracy | Precision | Recall | F1-Score |
|-------|----------|-----------|--------|----------|
| Logistic Regression | {results['Logistic Regression']['accuracy']*100:.2f}% | {results['Logistic Regression']['precision']:.4f} | {results['Logistic Regression']['recall']:.4f} | {results['Logistic Regression']['f1']:.4f} |
| Naive Bayes | {results['Naive Bayes']['accuracy']*100:.2f}% | {results['Naive Bayes']['precision']:.4f} | {results['Naive Bayes']['recall']:.4f} | {results['Naive Bayes']['f1']:.4f} |
| LSTM (BiLSTM) | {results['LSTM (BiLSTM)']['accuracy']*100:.2f}% | {results['LSTM (BiLSTM)']['precision']:.4f} | {results['LSTM (BiLSTM)']['recall']:.4f} | {results['LSTM (BiLSTM)']['f1']:.4f} |

## Key Findings
- LSTM outperforms classical models by learning contextual patterns
- Logistic Regression is fastest and most interpretable
- Naive Bayes is lightest but least accurate
- All models struggle with sarcasm and negation

## Figures
- reports/figures/model_comparison.png
- reports/figures/all_confusion_matrices.png
- reports/figures/lstm_training_curves.png
"""

with open('C:/data/BrandPulse-AI/reports/model_comparison_report.md', 'w') as f:
    f.write(report)

print("Report saved ")
print(report)